# 模型预训练

#### 几何增强版本: 仅随机水平/垂直翻转 + 随机 90° 旋转，无相位偏移、无噪声、无 weight_decay

## 导入库

In [ ]:
import os, sys
sys.path.insert(0, '/root/autodl-tmp/PINet_cpx_cl')
import random
import torch
import matplotlib.pyplot as plt
import torch.backends.cudnn as cudnn
import numpy as np
from torch import nn
from torch.utils.data import DataLoader
from src.model import PINet_cpx_v6
from src.mydataset import ComplexDataset_npy_plus, make_collate_fn
from src.myloss import ComplexLoss_amp_phs
from src.config import *
from src.utilities import diff_compute, device, compute_psnr_skimage_single

## 自定义 collate_fn: 仅几何增强（翻转+旋转）

只做随机水平/垂直翻转 + 随机 90° 旋转:
- 同步作用于 label（训练目标）和 label_shifted（衍射计算输入）
- 不做相位偏移、不做噪声注入

In [ ]:
def make_collate_fn_geometric(z_low=1.5, z_high=3, k=None, FX=None, FY=None,
                               device='cpu', diff_compute=None):
    def collate_fn(batch):
        label = torch.stack(batch)
        B = label.shape[0]

        # ---- 几何增强: 随机翻转 + 90° 旋转 ----
        if random.random() > 0.5:
            label = torch.flip(label, dims=[-1])
        if random.random() > 0.5:
            label = torch.flip(label, dims=[-2])
        k_rot = random.randint(0, 3)
        if k_rot > 0:
            label = torch.rot90(label, k_rot, dims=[-2, -1])

        label = label.to(device)

        # ---- 衍射传播 ----
        z = np.random.uniform(z_low*1e-3, z_high*1e-3)
        TF = np.exp(1j * z * np.sqrt(k**2 - (2*np.pi*FX)**2 - (2*np.pi*FY)**2))
        TF_torch = torch.from_numpy(TF).to(device).to(torch.complex64)
        diff = diff_compute(label, TF_torch).float()

        return diff, label, TF_torch
    return collate_fn

## 参数及路径设置

In [ ]:
torch.cuda.empty_cache()
cudnn.benchmark = True
print(f'Using device: {DEVICE}')

size = IMAGE_SIZE
z_low = Z_LOW
z_high = Z_HIGH
z = np.random.uniform(z_low*1e-3, z_high*1e-3)
TF = np.exp(1j * z * np.sqrt(k ** 2 - (2 * np.pi * FX) ** 2 - (2 * np.pi * FY) ** 2))
TF_torch = torch.from_numpy(TF).to(device).to(torch.complex64)
train_data_folder = os.path.join(DATA_DIR, 'train')
test_data_folder = os.path.join(DATA_DIR, 'val')
os.makedirs(OUTPUT_DIR, exist_ok=True)
log_name = f"training_log_size{size}_batchsize{BATCH_SIZE}_fold{FOLD_ITERS}_{Z_LOW}mm_to_{Z_HIGH}mm_dx_{dx}.txt"

## 导入数据集

In [ ]:
collate_fn = make_collate_fn_geometric(
    z_low=Z_LOW, z_high=Z_HIGH, k=k, FX=FX, FY=FY,
    device=device, diff_compute=diff_compute
)

dataset = ComplexDataset_npy_plus(train_data_folder, NUM_TRAIN_SAMPLES)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

collate_fn_test = make_collate_fn(
    z_low=Z_LOW, z_high=Z_HIGH, k=k, FX=FX, FY=FY,
    device=device, diff_compute=diff_compute
)
dataset_test = ComplexDataset_npy_plus(test_data_folder, NUM_VAL_SAMPLES)
dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=True, collate_fn=collate_fn_test)

##### 显示数据(衍射图像、目标振幅、目标相位)，确保数据集正确加载

In [ ]:
diff_tensor, label_tensor, tf = next(iter(dataloader_test))
print(f"shape of label: {label_tensor.shape}")
print(f"type of label: {label_tensor.dtype}")
print(f"shape of diff: {diff_tensor.shape}")
print(f"type of diff: {diff_tensor.dtype}")

print(f'amp max:{torch.max(torch.abs(label_tensor))}')
print(f'amp min:{torch.min(torch.abs(label_tensor))}')
print(f'phs max:{torch.max(torch.angle(label_tensor))}')
print(f'phs min:{torch.min(torch.angle(label_tensor))}')

plt.figure()
fig,axes = plt.subplots(1,3,figsize=(12,4))
axes[0].imshow(torch.abs(label_tensor).detach().cpu().squeeze(0).squeeze(0),cmap='gray')
axes[0].axis('off')
axes[0].set_title('Amp of Object')
axes[1].imshow(torch.angle(label_tensor).detach().cpu().squeeze(0).squeeze(0),cmap='gray')
axes[1].axis('off')
axes[1].set_title('Phs of Object')
axes[2].imshow(diff_tensor.detach().cpu().squeeze(0).squeeze(0),cmap='gray')
axes[2].axis('off')
axes[2].set_title('Diff of Object')
plt.show()

### 创建模型、损失、优化器

In [ ]:
pinet_cpx = PINet_cpx_v6(fold_iters=FOLD_ITERS).to(device)
print("This model has", sum(p.numel() for p in pinet_cpx.parameters()), "parameters.")

criterion = ComplexLoss_amp_phs(
    weight_amp=WEIGHT_AMP, weight_phs=WEIGHT_PHS, weight_diff=WEIGHT_DIFF
)
optimizer = torch.optim.Adam(pinet_cpx.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=LR_GAMMA)

In [ ]:
print(diff_tensor.shape)
pred, y_rec = pinet_cpx(diff_tensor.to(device), TF_torch)
plt.figure()
fig,axes = plt.subplots(1,3,figsize=(12,4))
axes[0].imshow(torch.abs(pred).detach().cpu().squeeze(0).squeeze(0),cmap='gray')
axes[0].axis('off')
axes[0].set_title('Amp of Pred')
axes[1].imshow(torch.angle(pred).detach().cpu().squeeze(0).squeeze(0),cmap='gray')
axes[1].axis('off')
axes[1].set_title('Phs of Pred')
axes[2].imshow(diff_tensor.detach().cpu().squeeze(0).squeeze(0),cmap='gray')
axes[2].axis('off')
axes[2].set_title('Diff of Object')
plt.show()
print(f"输入图像最大值: {diff_tensor.max()}")
print(f"输入图像最小值: {diff_tensor.min()}")
print(f"生成振幅最大值: {torch.abs(pred).max()}")
print(f"生成振幅最小值: {torch.abs(pred).min()}")
print(f"生成相位最大值: {torch.angle(pred).max()}")
print(f"生成相位最小值: {torch.angle(pred).min()}")

#### Checkpoint

In [ ]:
model_name = ''
checkpoint_path = os.path.join(OUTPUT_DIR, model_name)

def load_checkpoint(checkpoint_path, model, optimizer, scheduler):
    if os.path.isfile(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        epoch = checkpoint['epoch']
        loss = checkpoint['loss']
        print(f"Checkpoint loaded, starting from epoch {epoch + 1}")
        return epoch, loss
    else:
        print("No checkpoint found, starting from scratch.")
        return 0, None

## 模型训练

In [ ]:
from tqdm import tqdm
import logging

log_file_path = os.path.join(OUTPUT_DIR, log_name)
logging.basicConfig(
    filename=log_file_path,
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    force=True
)

start_epoch, _ = load_checkpoint(checkpoint_path, pinet_cpx, optimizer, scheduler)

loss_file_path = os.path.join(OUTPUT_DIR, "losses.txt")
if os.path.exists(loss_file_path):
    os.remove(loss_file_path)
with open(loss_file_path, "w") as f:
    f.write("Epoch Loss\n")

logging.info("")
logging.info("========== New Training Session (geometric aug only, no weight_decay) ==========")

for epoch in range(start_epoch, EPOCHS):
    running_loss = 0
    with tqdm(dataloader, desc=f"Epoch [{epoch+1}/{EPOCHS}]", unit="batch") as progress_bar:
        for diff, obj, TF_torch in progress_bar:
            diff = diff.to(device)
            obj = obj.to(device)
            pred, y_rec = pinet_cpx(diff, TF_torch)
            loss = criterion(pred, obj, diff, y_rec)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item(), lr=scheduler.get_last_lr()[0])

    scheduler.step()
    avg_loss = running_loss / len(dataloader)
    log_message = f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}"
    print(log_message)
    logging.info(log_message)
    with open(loss_file_path, "a") as f:
        f.write(f"{avg_loss:.4f}\n")

    torch.cuda.empty_cache()

    if (epoch + 1) % 10 == 0:
        model_name = f"model_pinet_size{size}_epoch_{epoch+1}_batchsize{BATCH_SIZE}_fold_iters{FOLD_ITERS}_{Z_LOW}mm_to_{Z_HIGH}mm_dx_{dx}_geometric.pth"
        model_path = os.path.join(OUTPUT_DIR, model_name)

        total_psnr_amp = 0
        total_psnr_phs = 0
        total_samples = 0

        with torch.no_grad():
            for diff, obj, TF_torch in dataloader_test:
                diff = diff.to(device)
                obj = obj.to(device)
                pred, y_rec = pinet_cpx(diff, TF_torch)

                pred_amp = torch.abs(pred)
                pred_phs = torch.angle(pred)
                obj_amp = torch.abs(obj)
                obj_phs = torch.angle(obj)

                psnr_amp = compute_psnr_skimage_single(pred_amp, obj_amp)
                psnr_phs = compute_psnr_skimage_single(pred_phs, obj_phs)

                total_psnr_amp += psnr_amp
                total_psnr_phs += psnr_phs
                total_samples += 1

        avg_psnr_amp = total_psnr_amp / total_samples
        avg_psnr_phs = total_psnr_phs / total_samples
        psnr_message = f'PSNR_amp:{avg_psnr_amp:.2f}dB,PSNR_phs:{avg_psnr_phs:.2f}dB'
        print(psnr_message)
        logging.info(psnr_message)

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': pinet_cpx.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_loss,
        }, model_path)

        torch.cuda.empty_cache()
        print(f"Model saved at {model_path}")